# Activation Steering

vLLM-Hook is an extensible framework that aims to allow selective access to model internals during inference.
In this notebook, we demonstrate how vLLM-Hook enables **Activation Steering** for controlled generation.

**Paper**: [Improving Instruction-Following in Language Models through Activation Steering](https://arxiv.org/abs/2410.12877).<br />
**Authors**: Alessandro Stolfo, Vidhisha Balachandran, Safoora Yousefi, Eric Horvitz, Besmira Nushi <br />
**"TL;DR"**: Activation steering allows you to bias the model's behavior by nudging internal activations in specific directions. In this paper, authors focus on instruction following capability and compute the steering vectors as the difference in activations between inputs with and without instructions. 

### Installation

If running this from a new environment, please use the cell below to install `vllm_hook_plugins`. Update the path/command to match your environment.<br />
The following block is not necessary if running this notebook from an environment where the package has already been installed.

In [ ]:
from pathlib import Path
import sys

# vllm_hooks/notebooks/
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT/"vllm_hook_plugins"
REQ_FILE = REPO_ROOT/"requirement.txt"

print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

%pip install -e "{PKG_DIR}"

if REQ_FILE.exists():
    %pip install -r "{REQ_FILE}"
else:
    print("⚠️ requirements.txt not found at", REQ_FILE)


### Importing the Hook-Enabled LLM
The plugin provides its own LLM wrapper that behaves like vllm.LLM (`from vllm import LLM`) but adds support for hooks and instrumentation.
We import it here:

In [1]:
from vllm_hook_plugins import HookLLM

/proj/dmfexp/irene/miniconda3/envs/vllm_hook_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Environment & multiprocessing setup

In [2]:
import os
import multiprocessing as mp
import torch
from pathlib import Path
from vllm import SamplingParams
mp.set_start_method("spawn", force=True)
os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Steering vector paths in model_configs/.../*.json are relative to the repo
# root, so chdir there before constructing HookLLM.
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

### Initialize `HookLLM`
Before we create the LLM instance, we need to specify the model and data type:

In [3]:
cache_dir = './cache'  # Specify cache dir
model = 'microsoft/Phi-3-mini-4k-instruct'

dtype_map = {
    'microsoft/Phi-3-mini-4k-instruct': 'auto',
}

We also need to provide a config file that specifies how activations are steered (e.g., which layers to intervene on, which token to intervene, what direction vectors to apply, etc.).<br />
In the following example, we apply activation steering at the 15th layer, apply the steering at all positions (as opposed to only at the start of the decoding process), and along the direction given in `vector_path`:

In [4]:
import json

json_path = Path("model_configs/activation_steer/Phi-3-mini-4k-instruct.json")  # adjust path

with open(json_path, "r") as f:
    config = json.load(f)

# print(config)

Inside `steer_hook_act` we defined the activation steering behavior during model inference.
Now, we initialize the llm:

In [5]:
llm = HookLLM(
    model=model,
    worker_name="steer_hook_act",
    config_file=json_path,
    download_dir=cache_dir,
    gpu_memory_utilization=0.7,
    trust_remote_code=True,
    dtype=dtype_map[model],
    enable_prefix_caching=True,
    enable_hook=True,
)

INFO 06-01 23:12:13 [utils.py:233] non-default args: {'trust_remote_code': True, 'download_dir': './cache', 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'enforce_eager': True, 'worker_extension_cls': 'vllm_hook_plugins.workers.steer_activation_worker.SteerHookActWorker', 'model': 'microsoft/Phi-3-mini-4k-instruct'}
WARNING 06-01 23:12:13 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1


2026-06-01 23:12:13,686	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 06-01 23:12:13 [model.py:549] Resolved architecture: Phi3ForCausalLM
INFO 06-01 23:12:13 [model.py:1678] Using max model len 4096
INFO 06-01 23:12:13 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 06-01 23:12:13 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 06-01 23:12:13 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-01 23:12:13 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-01 23:12:14 [vllm.py:1025] Cudagraph is disabled under eager mode
INFO 06-01 23:12:14 [compilation.py:292] Enabled custom fusions: norm_quant, act_quant
(EngineCore pid=3981266) INFO 06-01 23:12:23 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='microsoft/Phi-3-mini-4k-instruct', speculative_config=None, tokenizer='mi

(EngineCore pid=3981266) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.24s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.74s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.81s/it]
(EngineCore pid=3981266) 


(EngineCore pid=3981266) INFO 06-01 23:12:29 [default_loader.py:384] Loading weights took 3.72 seconds
(EngineCore pid=3981266) INFO 06-01 23:12:30 [gpu_model_runner.py:4820] Model loading took 7.12 GiB memory and 4.501628 seconds
(EngineCore pid=3981266) INFO 06-01 23:12:31 [gpu_worker.py:436] Available KV cache memory: 47.02 GiB
(EngineCore pid=3981266) INFO 06-01 23:12:31 [kv_cache_utils.py:1319] GPU KV cache size: 128,384 tokens
(EngineCore pid=3981266) INFO 06-01 23:12:31 [kv_cache_utils.py:1324] Maximum concurrency for 4,096 tokens per request: 31.22x


(EngineCore pid=3981266) 2026-06-01 23:12:31,902 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=3981266) 2026-06-01 23:12:31,916 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends


(EngineCore pid=3981266) INFO 06-01 23:12:32 [core.py:283] init engine (profile, create kv cache, warmup model) took 1.70 seconds
(EngineCore pid=3981266) INFO 06-01 23:12:33 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=3981266) WARNING 06-01 23:12:33 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=3981266) WARNING 06-01 23:12:33 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=3981266) INFO 06-01 23:12:33 [vllm.py:1025] Cudagraph is disabled under eager mode
(EngineCore pid=3981266) INFO 06-01 23:12:33 [compilation.py:292] Enabled custom fusions: norm_quant, act_quant


### Test case
In the following, we show a test case and compare generations **with** and **without** activation steering.

**Note**: Users should swap the example configs with their own to show desirable performance. The following is for pipeline illustration only.

In [6]:
test_cases = [
    "Write a dialogue between two people, one is dressed up in a ball gown and the other is dressed down in sweats. The two are going to a nightly event. Your answer must contain exactly 3 bullet points in the markdown format (use \"* \" to indicate each bullet) such as:\n* This is the first point.\n* This is the second point.",
    "What is the difference between the 13 colonies and the other British colonies in North America? Your answer must contain exactly 6 bullet point in Markdown using the following format:\n* Bullet point one.\n* Bullet point two.\n...\n* Bullet point fix."
]

Before we start, we define the sampling parameters:

**Note**: token 32007 is phi-specific, refer to the original huggingface implementation for details https://github.com/microsoft/llm-steer-instruct/blob/main/utils/generation_utils.py.

- Option 1: we can use a uniform sampling parameters as follows

In [7]:
# sampling_params = SamplingParams(
#     temperature=0.0,                       
#     max_tokens=2048,
#     stop_token_ids=[llm.tokenizer.eos_token_id, 32007],  
# )

- Option 2: we can vary the steering settings per request by passing a list of `SamplingParams` aligned with the prompts. Each entry's `extra_args["steer"]` carries the full steering config for that request, so different prompts in the **same batch** can use different coefficients, methods, optimal layers, etc. Below, the first prompt uses the JSON config as-is, while the second prompt overrides `method` and `coefficient` (everything else inherited from the JSON config).

In [8]:
base_steer = config["steering"]  # full default config from the JSON

sampling_params_list = [
    SamplingParams(
        temperature=0.0,
        max_tokens=2048,
        stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
    ),
    SamplingParams(
        temperature=0.0,
        max_tokens=2048,
        stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
        extra_args={"steer": {**base_steer, "method": "add_vector", "coefficient": 10}},
    ),
]

Next we:
1. Apply chat template on each test case,
2. Generate all prompts in one batched call with their per-request `SamplingParams`. With activation steering enabled (`use_hook=True`, default), each prompt's hook reads its own `extra_args["steer"]` so different prompts in the same batch use different steer settings,
3. Reset the prefix cache to ensure the baseline generation does not reuse steered cache,
4. Generate again with `use_hook=False` to obtain the baseline output.

In [9]:
examples = [
    llm.tokenizer.apply_chat_template(
        [{"role": "user", "content": case}], add_generation_prompt=True, tokenize=False
    )
    for case in test_cases
]

outputs = llm.generate(examples, sampling_params_list)

llm.llm_engine.reset_prefix_cache()

outputs_original = llm.generate(examples, sampling_params_list, use_hook=False)

Processed prompts: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.41s/it, est. speed input: 51.73 toks/s, output: 128.26 toks/s]


(EngineCore pid=3981266) Installed 32 steering hooks on layers: ['model.layers.0', 'model.layers.1', 'model.layers.2', 'model.layers.3', 'model.layers.4', 'model.layers.5', 'model.layers.6', 'model.layers.7', 'model.layers.8', 'model.layers.9', 'model.layers.10', 'model.layers.11', 'model.layers.12', 'model.layers.13', 'model.layers.14', 'model.layers.15', 'model.layers.16', 'model.layers.17', 'model.layers.18', 'model.layers.19', 'model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28', 'model.layers.29', 'model.layers.30', 'model.layers.31']
(EngineCore pid=3981266) Hooks installed successfully
(EngineCore pid=3981266) INFO 06-01 23:12:36 [block_pool.py:472] Successfully reset prefix cache


Processed prompts: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.04s/it, est. speed input: 70.09 toks/s, output: 156.98 toks/s]


Finally we can print out the results as follows:

In [10]:
for steered, original in zip(outputs, outputs_original):
    print("=" * 100)
    steered_text = steered.outputs[0].text
    print("\n[With activation steering]\n")
    print(steered_text)
    
    baseline_text = original.outputs[0].text
    print("\n[Without activation steering]\n")
    print(baseline_text)


[With activation steering]

 * The woman in the ball gown is wearing a stunning, floor-length dress with intricate lace details and matching high heels.
* The man in the sweats is wearing a comfortable, casual t-shirt and sweatpants, with sneakers on his feet.
* They both realize they are going to a nightly event and start discussing what they should wear to make a good impression.

[Without activation steering]

 * The woman in the ball gown is excitedly discussing the upcoming event with her friend, who is dressed in casual sweats.
* The friend in sweats expresses their discomfort with the formal attire, but the woman in the ball gown reassures them that they will have a great time regardless of what they wear.
* The two friends make plans to meet at the event, with the woman in the ball gown promising to help her friend find a comfortable and stylish outfit for the occasion.

[With activation steering]

 * The 13 colonies, also known as the Thirteen Colonies, were British colonies 